# Notebook 1 — ABM: Can Parameters Emerge from Practice?

Notebook 0 established the formal properties of Berman's SAW model and defined
five testable criteria that any set of practice-grounded parameters must satisfy.
This notebook asks whether those parameters can actually *emerge* from simulated
legal practice.

**Method.** An Agent-Based Model (ABM) simulates a bench of judges deciding
cases over time. Each judge has a methodology (textualist, purposivist,
pragmatist, originalist) that determines which principles they tend to invoke.
Through iterated case decisions and selection pressure, principle weights
emerge from the pattern of invocation rather than being stipulated.

**Experiments:**
1. **Three-condition emergence** — Does bimodal principle differentiation emerge,
   and does it depend on the fitness criterion (Hart-adjacent vs. Berman)?
2. **Sensitivity analyses** — Do the results depend on agent count, invocation
   tendency spread, or selection cutoff?
3. **Emergent activation structure** — Does the ABM produce realistic correlation
   patterns between principles?
4. **Cross-validation** — Do the emergent weights pass Notebook 0's five tests?
5. **SAW fidelity** — Does the ABM's emergent behavior match the SAW formula?

**Runtime:** approximately 40–55 minutes on Colab CPU with High-RAM.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Setup

In [2]:
%matplotlib inline
!pip install numpy pandas matplotlib seaborn scipy scikit-learn diptest -q
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.7/222.7 kB 5.9 MB/s eta 0:00:00
Dependencies installed.


## Core Model (from Notebook 0)

The code below is copied verbatim from Notebook 0. It provides the SAW model,
principle generators, and statistical helpers that the cross-validation
experiments require. Nothing here is new — it is reproduced so this notebook
is self-contained and independently runnable.

In [3]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from collections import defaultdict
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'figure.dpi': 100, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
})

# --- Notebook 0 core model (verbatim) ---

@dataclass
class Principle:
    name: str
    weight: float
    description: str = ""
    category: str = "general"
    def __post_init__(self):
        assert 0 <= self.weight <= 1, f"Weight must be in [0,1], got {self.weight}"

@dataclass
class Case:
    name: str
    facts: Dict[str, float]

class BermanModel:
    def __init__(self, principles: List[Principle], threshold: float = 0.2):
        self.principles = {p.name: p for p in principles}
        self.threshold = threshold

    def net_force(self, case: Case, weight_noise: float = 0.0) -> float:
        f = 0.0
        for name, activation in case.facts.items():
            if name not in self.principles:
                continue
            w = self.principles[name].weight
            if weight_noise > 0:
                w = np.clip(w + np.random.normal(0, weight_noise), 0, 1)
            f += w * activation
        return f

    def is_determinate(self, case: Case, weight_noise: float = 0.0) -> bool:
        return abs(self.net_force(case, weight_noise)) >= self.threshold

def make_principles(n, distribution, seed=0):
    rng = np.random.RandomState(seed)
    if distribution == 'uniform':
        weights = rng.uniform(0.60, 0.70, n)
    elif distribution == 'moderate':
        weights = rng.uniform(0.50, 0.80, n)
    elif distribution == 'high':
        weights = rng.uniform(0.30, 0.95, n)
    else:
        raise ValueError(distribution)
    return [Principle(f"P{i}", float(w)) for i, w in enumerate(weights)]

def random_case(principle_names, rng=None):
    if rng is None: rng = np.random
    return Case("synth", {p: float(rng.uniform(-1, 1)) for p in principle_names})

def bootstrap_ci(values, n_boot=2000, ci=95):
    arr = np.array(values, dtype=float)
    means = np.array([np.mean(np.random.choice(arr, len(arr), replace=True))
                      for _ in range(n_boot)])
    lo = np.percentile(means, (100 - ci) / 2)
    hi = np.percentile(means, 100 - (100 - ci) / 2)
    return float(np.mean(arr)), float(lo), float(hi)

def find_crossing(curve, levels, threshold):
    for j in range(1, len(curve)):
        if curve[j] < threshold and curve[j-1] >= threshold:
            frac = (threshold - curve[j-1]) / (curve[j] - curve[j-1])
            return float(levels[j-1] + frac * (levels[j] - levels[j-1]))
    return float(levels[-1])

print("Core model (from Notebook 0) loaded.")

Core model (from Notebook 0) loaded.


## ABM Configuration

The invocation tendencies define each methodology's relationship to each
principle. The design creates two natural tiers:

- **Cross-cutting principles** (text, original, precedent, structure, federalism):
  invoked at moderate-to-high rates by 2–3 methodologies.
- **Niche principles** (dignity, equality, liberty, pragmatic): invoked primarily
  by one methodology.

Whether this two-tier structure produces a bimodal *weight* distribution through
practice is the question Experiment 1 tests. The specific numbers below (0.95,
0.04, etc.) represent the "wide spread" configuration. Experiment 2 tests
narrower spreads to check whether the result depends on the extremity of
these numbers.

In [4]:
# ABM CONFIGURATION
# ============================================================================

PRINCIPLES_DEFAULT = [
    'text', 'original', 'structure', 'precedent',
    'dignity', 'equality', 'liberty', 'federalism', 'pragmatic'
]

METHODOLOGIES = ['textualist', 'purposivist', 'pragmatist', 'originalist']

def make_invocation_tendencies(spread='wide'):
    """
    Generate methodology-to-principle invocation tendencies.

    spread:
      'wide'   — HIGH 0.85-0.95, LOW 0.02-0.08 (strong specialisation)
      'medium' — HIGH 0.70-0.80, LOW 0.15-0.25 (moderate specialisation)
      'narrow' — HIGH 0.60-0.70, LOW 0.30-0.40 (weak specialisation)

    The spread parameter is tested in Experiment 2 to check whether
    bimodality depends on extreme specialisation.
    """
    if spread == 'wide':
        hi, lo = (0.85, 0.95), (0.02, 0.08)
        md_range = (0.35, 0.65)
    elif spread == 'medium':
        hi, lo = (0.70, 0.80), (0.15, 0.25)
        md_range = (0.40, 0.60)
    elif spread == 'narrow':
        hi, lo = (0.60, 0.70), (0.30, 0.40)
        md_range = (0.42, 0.58)
    else:
        raise ValueError(spread)

    # Which principles are HIGH/LOW/MEDIUM for each methodology
    # This structure is fixed; only the numerical ranges change with spread.
    structure = {
        'textualist': {
            'high': ['text', 'original'],
            'med':  ['structure', 'federalism', 'precedent'],
            'low':  ['dignity', 'equality', 'liberty', 'pragmatic'],
        },
        'purposivist': {
            'high': ['dignity', 'equality', 'liberty'],
            'med':  ['precedent', 'pragmatic', 'structure'],
            'low':  ['text', 'original', 'federalism'],
        },
        'pragmatist': {
            'high': ['pragmatic', 'precedent'],
            'med':  ['structure', 'liberty', 'federalism', 'text'],
            'low':  ['dignity', 'equality', 'original'],
        },
        'originalist': {
            'high': ['original', 'text', 'federalism'],
            'med':  ['structure', 'precedent'],
            'low':  ['dignity', 'equality', 'liberty', 'pragmatic'],
        },
    }

    tendencies = {}
    for method, groups in structure.items():
        t = {}
        for p in PRINCIPLES_DEFAULT:
            if p in groups['high']:
                t[p] = np.random.uniform(*hi)
            elif p in groups['low']:
                t[p] = np.random.uniform(*lo)
            else:
                t[p] = np.random.uniform(*md_range)
        tendencies[method] = t

    return tendencies

INVOCATION_TENDENCIES = make_invocation_tendencies('wide')

print("ABM configuration loaded.")
print(f"  Principles: {len(PRINCIPLES_DEFAULT)}")
print(f"  Methodologies: {len(METHODOLOGIES)}")
print(f"  Spread: wide (0.85-0.95 / 0.02-0.08)")

ABM configuration loaded.
  Principles: 9
  Methodologies: 4
  Spread: wide (0.85-0.95 / 0.02-0.08)


## Agent and Legal System

**Agent:** Each judge has a methodology that determines invocation tendencies.
Invocation decisions blend the agent's own tendency (85%) with the system-wide
weight (15%), preserving methodological identity while allowing limited social
learning. Agents update their tendencies based on whether they were "successful"
under the fitness criterion.

**LegalSystem:** Simulates a bench deciding cases. System-wide principle weights
update every 100 cases based on relative invocation frequency. Selection pressure
replaces the bottom 20% of agents every 500 cases with mutated copies of top
performers. An optional median threshold separates high-frequency principles from
low-frequency ones.

The fitness criterion is configurable:
- **'consensus'** — agent succeeds when its force direction matches the panel
  majority. This is a Hart-adjacent picture where legal practice rewards agreement.
- **'berman'** — agent succeeds when its |force| exceeds the panel median. This
  reflects Berman's account that legal reasoning succeeds by producing decisive
  arguments.

In [5]:
# AGENT AND LEGAL SYSTEM
# ============================================================================

@dataclass
class Agent:
    agent_id: int
    methodology: str
    invocation_tendencies: Dict[str, float] = field(default_factory=dict)
    success_count: int = 0
    total_decisions: int = 0

    @classmethod
    def create(cls, agent_id, methodology, tendencies_map):
        """Create an agent with tendencies from a given map, plus jitter."""
        base = tendencies_map[methodology].copy()
        jitter = {p: np.clip(v + np.random.normal(0, 0.03), 0.02, 0.98)
                  for p, v in base.items()}
        return cls(agent_id=agent_id, methodology=methodology,
                   invocation_tendencies=jitter)

    def decide_case(self, activations, system_weights):
        invoked = []
        net_force = 0.0
        for p, activation in activations.items():
            base = self.invocation_tendencies.get(p, 0.5)
            sw = system_weights.get(p, 0.5)
            prob = 0.85 * base + 0.15 * sw
            if abs(activation) > 0.3:
                prob = min(prob * 1.15, 0.99)
            if np.random.random() < prob:
                invoked.append(p)
                net_force += sw * activation
        return invoked, net_force

    def update_tendencies(self, invoked, was_successful, lr=0.04):
        if len(invoked) > 4:
            lr *= 0.5
        for p in self.invocation_tendencies:
            if p in invoked:
                delta = lr if was_successful else -lr
            else:
                delta = -lr * 0.08
            self.invocation_tendencies[p] = np.clip(
                self.invocation_tendencies[p] + delta, 0.02, 0.98)


class LegalSystem:
    def __init__(self, n_agents=100, principles=None,
                 use_median_threshold=False, fitness_criterion='berman',
                 selection_percentile=50, seed=42,
                 invocation_tendencies=None):
        """
        Parameters:
          n_agents: number of judges
          principles: list of principle names
          use_median_threshold: if True, principles below cutoff → weight 0
          fitness_criterion: 'berman' or 'consensus'
          selection_percentile: cutoff for principle death (default 50 = median)
          seed: random seed
          invocation_tendencies: override global tendencies (for sensitivity tests)
        """
        np.random.seed(seed)
        self.principles = principles or PRINCIPLES_DEFAULT
        self.use_median_threshold = use_median_threshold
        self.fitness_criterion = fitness_criterion
        self.selection_percentile = selection_percentile
        self.n_agents = n_agents

        # Use provided tendencies or fall back to global default
        tend = invocation_tendencies if invocation_tendencies is not None else INVOCATION_TENDENCIES

        # Build agents with equal methodology distribution
        methods = (METHODOLOGIES * (n_agents // len(METHODOLOGIES) + 1))[:n_agents]
        np.random.shuffle(methods)
        self.agents = [Agent.create(i, m, tend) for i, m in enumerate(methods)]

        # System state
        self.invocation_counts = {p: 0 for p in self.principles}
        self.total_invocations = 0
        self.system_weights = {p: 0.5 for p in self.principles}
        self.case_count = 0

        # History
        self.raw_freq_history = []
        self.weight_history = []
        self.force_outcome_log = []

    def _compute_weights(self):
        if self.total_invocations == 0:
            return {p: 0.5 for p in self.principles}
        raw = {p: self.invocation_counts[p] / self.total_invocations
               for p in self.principles}
        max_r = max(raw.values()) or 1.0
        norm = {p: v / max_r for p, v in raw.items()}

        if self.use_median_threshold:
            vals = list(norm.values())
            cutoff = np.percentile(vals, self.selection_percentile)
            norm = {p: (v if v >= cutoff else 0.0) for p, v in norm.items()}
            max_s = max(norm.values()) or 1.0
            norm = {p: v / max_s for p, v in norm.items()}
        return norm

    def _get_raw_normalized(self):
        if self.total_invocations == 0:
            return {p: 0.5 for p in self.principles}
        raw = {p: self.invocation_counts[p] / self.total_invocations
               for p in self.principles}
        max_r = max(raw.values()) or 1.0
        return {p: v / max_r for p, v in raw.items()}

    def _apply_selection(self):
        tend = {m: {} for m in METHODOLOGIES}
        # Rebuild tendency map from current top agents for mutation
        rated = sorted(self.agents,
                       key=lambda a: a.success_count / max(a.total_decisions, 1))
        n_rep = max(1, self.n_agents // 5)
        tops = rated[-n_rep:]
        for agent in rated[:n_rep]:
            tmpl = np.random.choice(tops)
            agent.methodology = tmpl.methodology
            for p in agent.invocation_tendencies:
                agent.invocation_tendencies[p] = np.clip(
                    tmpl.invocation_tendencies[p] + np.random.normal(0, 0.08),
                    0.02, 0.98)
            agent.success_count = 0
            agent.total_decisions = 0

    def simulate(self, n_cases=5000, panel_size=9,
                 update_every=100, record_every=200,
                 selection_every=500):
        for c in range(n_cases):
            self.case_count += 1
            activations = {p: np.random.uniform(-1, 1) for p in self.principles}
            panel = np.random.choice(self.agents, size=panel_size, replace=False)

            decisions = []
            for agent in panel:
                invoked, force = agent.decide_case(activations, self.system_weights)
                for p in invoked:
                    self.invocation_counts[p] += 1
                    self.total_invocations += 1
                decisions.append((invoked, force, agent))

            forces = [f for _, f, _ in decisions]
            mean_force = float(np.mean(forces))
            is_det = abs(mean_force) > 0.05
            self.force_outcome_log.append((mean_force, is_det))

            if self.fitness_criterion == 'berman':
                abs_forces = [abs(f) for f in forces]
                panel_median_force = float(np.median(abs_forces))
                for invoked, force, agent in decisions:
                    success = abs(force) >= panel_median_force
                    agent.total_decisions += 1
                    if success:
                        agent.success_count += 1
                    agent.update_tendencies(invoked, success)
            else:
                pos = sum(1 for f in forces if f > 0.05)
                neg = sum(1 for f in forces if f < -0.05)
                consensus = 'pos' if pos > neg else ('neg' if neg > pos else 'tie')
                for invoked, force, agent in decisions:
                    d = 'pos' if force > 0.05 else ('neg' if force < -0.05 else 'tie')
                    success = (d == consensus)
                    agent.total_decisions += 1
                    if success:
                        agent.success_count += 1
                    agent.update_tendencies(invoked, success)

            if (c + 1) % update_every == 0:
                self.system_weights = self._compute_weights()
            if (c + 1) % record_every == 0:
                raw = self._get_raw_normalized()
                self.raw_freq_history.append((c + 1, raw.copy()))
                self.weight_history.append((c + 1, self.system_weights.copy()))
            if (c + 1) % selection_every == 0 and c > 0:
                self._apply_selection()

        return self._summarize()

    def _summarize(self):
        raw = self._get_raw_normalized()
        raw_vals = np.array(list(raw.values()))
        cv_raw = float(np.std(raw_vals) / np.mean(raw_vals)) if np.mean(raw_vals) > 0 else 0

        km = KMeans(n_clusters=2, random_state=0, n_init=10)
        labels = km.fit_predict(raw_vals.reshape(-1, 1))
        centers = km.cluster_centers_.flatten()
        hi_idx = int(centers[1] > centers[0])
        hi = raw_vals[labels == hi_idx]
        lo = raw_vals[labels != hi_idx]
        gap_raw = float(hi.min() - lo.max()) if len(hi) > 0 and len(lo) > 0 else 0.0

        w_vals = np.array(list(self.system_weights.values()))
        cv_w = float(np.std(w_vals) / np.mean(w_vals)) if np.mean(w_vals) > 0 else 0

        return {
            'raw_freqs': raw,
            'system_weights': self.system_weights.copy(),
            'cv_raw': cv_raw,
            'gap_raw': gap_raw,
            'cv_weights': cv_w,
            'n_survivors': int(np.sum(w_vals > 0.01)),
            'raw_freq_history': self.raw_freq_history,
            'weight_history': self.weight_history,
            'force_outcome_log': self.force_outcome_log,
        }

print("Agent and LegalSystem classes ready.")

Agent and LegalSystem classes ready.


## Experiment 1 — Three-Condition Emergence

**Question:** Does bimodal principle differentiation emerge from practice, and
does the answer depend on what the legal system selects for?

Three conditions run in parallel with the same seeds:
- **A (consensus fitness, no threshold):** Hart-adjacent — agents rewarded for
  matching panel majority direction.
- **B (Berman fitness, no threshold):** Agents rewarded for producing above-median
  |force|. Selects for decisive argumentation.
- **C (Berman fitness, with median threshold):** Full Berman model with principle
  death. The threshold formalises the gap already present in B.

The A→B comparison is the computational instantiation of the Hart/Berman debate.
If B produces stronger bimodality, Berman's mechanism generates sharper
differentiation. Both results are reported; the gap is tested for significance.

**Statistical tests:**
- Hartigan's dip test for formal unimodality rejection.
- Paired Wilcoxon signed-rank test on A vs. B gap distributions.

In [6]:
# EXPERIMENT 1: THREE-CONDITION EMERGENCE
# ============================================================================

import diptest

def experiment_1_emergence(n_seeds=20, n_cases=5000):
    print("\n" + "=" * 70)
    print("EXPERIMENT 1: Three-Condition Emergence")
    print("  A: consensus fitness, no threshold (Hart-adjacent)")
    print("  B: Berman fitness,   no threshold")
    print("  C: Berman fitness,   with median threshold")
    print(f"  Seeds: {n_seeds}, Cases per seed: {n_cases}")
    print("=" * 70)

    results_A, results_B, results_C = [], [], []
    dip_pvals_A, dip_pvals_B = [], []

    for seed in range(n_seeds):
        # Condition A: consensus, no threshold
        lsys_A = LegalSystem(use_median_threshold=False,
                             fitness_criterion='consensus', seed=seed)
        r_A = lsys_A.simulate(n_cases=n_cases)
        results_A.append(r_A)

        # Condition B: berman, no threshold
        lsys_B = LegalSystem(use_median_threshold=False,
                             fitness_criterion='berman', seed=seed)
        r_B = lsys_B.simulate(n_cases=n_cases)
        results_B.append(r_B)

        # Condition C: berman, with threshold
        lsys_C = LegalSystem(use_median_threshold=True,
                             fitness_criterion='berman', seed=seed)
        r_C = lsys_C.simulate(n_cases=n_cases)
        results_C.append(r_C)

        # Hartigan's dip test on raw frequencies
        vals_A = np.array(list(r_A['raw_freqs'].values()))
        vals_B = np.array(list(r_B['raw_freqs'].values()))
        _, pval_A = diptest.diptest(vals_A)
        _, pval_B = diptest.diptest(vals_B)
        dip_pvals_A.append(pval_A)
        dip_pvals_B.append(pval_B)

        if (seed + 1) % 5 == 0:
            print(f"  Seed {seed+1:2d}/{n_seeds} | "
                  f"A gap={r_A['gap_raw']:.3f} dip_p={pval_A:.3f} | "
                  f"B gap={r_B['gap_raw']:.3f} dip_p={pval_B:.3f} | "
                  f"C cv_w={r_C['cv_weights']:.3f}")

    # Summary statistics
    def summarise(label, results, use_cv_weights=False):
        gaps = [r['gap_raw'] for r in results]
        cvs = [r['cv_weights'] if use_cv_weights else r['cv_raw'] for r in results]
        bimodal = sum(1 for g in gaps if g > 0.05)
        m_g, lo_g, hi_g = bootstrap_ci(gaps)
        m_c, lo_c, hi_c = bootstrap_ci(cvs)
        print(f"\n  {label}:")
        print(f"    Gap:  {m_g:.4f} [{lo_g:.4f}, {hi_g:.4f}]  "
              f"bimodal: {bimodal}/{n_seeds}")
        print(f"    CV:   {m_c:.4f} [{lo_c:.4f}, {hi_c:.4f}]")
        return gaps, cvs

    gaps_A, cvs_A = summarise("Condition A (consensus, no threshold)", results_A)
    gaps_B, cvs_B = summarise("Condition B (Berman, no threshold)", results_B)
    gaps_C, cvs_C = summarise("Condition C (Berman, with threshold)",
                              results_C, use_cv_weights=True)

    # Dip test summary
    reject_A = sum(1 for p in dip_pvals_A if p < 0.05)
    reject_B = sum(1 for p in dip_pvals_B if p < 0.05)
    print(f"\n  Hartigan's dip test (rejects unimodality at p<0.05):")
    print(f"    Condition A: {reject_A}/{n_seeds} seeds reject unimodality")
    print(f"    Condition B: {reject_B}/{n_seeds} seeds reject unimodality")

    # Paired Wilcoxon: A vs B
    stat, pval_AB = stats.wilcoxon(gaps_B, gaps_A, alternative='greater')
    diff_AB = np.mean(gaps_B) - np.mean(gaps_A)
    print(f"\n  A→B gap lift (Berman fitness effect):")
    print(f"    Mean Δgap = {diff_AB:+.4f}")
    print(f"    Wilcoxon p = {pval_AB:.4f}")
    if pval_AB < 0.05:
        print(f"    → Berman fitness produces significantly stronger bimodality.")
    else:
        print(f"    → No significant difference between fitness criteria.")

    # B→C lift
    diff_BC = np.mean(cvs_C) - np.mean(cvs_B)
    print(f"\n  B→C CV lift (threshold effect): {diff_BC:+.4f}")

    return results_A, results_B, results_C, dip_pvals_A, dip_pvals_B

results_A, results_B, results_C, dip_A, dip_B = experiment_1_emergence(n_seeds=20)


EXPERIMENT 1: Three-Condition Emergence
  A: consensus fitness, no threshold (Hart-adjacent)
  B: Berman fitness,   no threshold
  C: Berman fitness,   with median threshold
  Seeds: 20, Cases per seed: 5000
  Seed  5/20 | A gap=0.074 dip_p=0.922 | B gap=0.645 dip_p=0.001 | C cv_w=0.895
  Seed 10/20 | A gap=0.140 dip_p=0.476 | B gap=0.724 dip_p=0.000 | C cv_w=0.895
  Seed 15/20 | A gap=0.142 dip_p=0.152 | B gap=0.704 dip_p=0.000 | C cv_w=0.895
  Seed 20/20 | A gap=0.124 dip_p=0.696 | B gap=0.711 dip_p=0.000 | C cv_w=0.895

  Condition A (consensus, no threshold):
    Gap:  0.1087 [0.0912, 0.1245]  bimodal: 19/20
    CV:   0.1317 [0.1248, 0.1382]

  Condition B (Berman, no threshold):
    Gap:  0.7024 [0.6866, 0.7184]  bimodal: 20/20
    CV:   0.6481 [0.6405, 0.6563]

  Condition C (Berman, with threshold):
    Gap:  0.6047 [0.5835, 0.6254]  bimodal: 20/20
    CV:   0.8947 [0.8947, 0.8947]

  Hartigan's dip test (rejects unimodality at p<0.05):
    Condition A: 0/20 seeds reject unimod

In [7]:
# EXPERIMENT 1 — VISUALISATION
# ============================================================================

fig = plt.figure(figsize=(18, 10))
fig.suptitle('Experiment 1: Three-Condition ABM Emergence', fontsize=14, fontweight='bold')

# Panel 1: Raw frequencies A vs B (seed 0)
ax1 = fig.add_subplot(2, 3, 1)
raw_A = results_A[0]['raw_freqs']
raw_B = results_B[0]['raw_freqs']
ps = sorted(raw_B.keys(), key=lambda p: raw_B[p], reverse=True)
x = np.arange(len(ps))
w = 0.35
ax1.barh(x + w/2, [raw_A[p] for p in ps], w, label='A: consensus', color='#aec7e8', edgecolor='black')
ax1.barh(x - w/2, [raw_B[p] for p in ps], w, label='B: Berman', color='#1f77b4', edgecolor='black')
ax1.set_yticks(x)
ax1.set_yticklabels(ps, fontsize=8)
ax1.set_title('Raw Frequencies: A vs B\n(Seed 0)', fontweight='bold')
ax1.legend(fontsize=8)

# Panel 2: Gap distributions
ax2 = fig.add_subplot(2, 3, 2)
gaps_A_all = [r['gap_raw'] for r in results_A]
gaps_B_all = [r['gap_raw'] for r in results_B]
bins = np.linspace(min(gaps_A_all + gaps_B_all) - 0.01,
                   max(gaps_A_all + gaps_B_all) + 0.01, 15)
ax2.hist(gaps_A_all, bins=bins, alpha=0.6, color='#aec7e8', edgecolor='black',
         label=f'A (μ={np.mean(gaps_A_all):.3f})')
ax2.hist(gaps_B_all, bins=bins, alpha=0.6, color='#1f77b4', edgecolor='black',
         label=f'B (μ={np.mean(gaps_B_all):.3f})')
ax2.axvline(0.05, color='red', ls='--', lw=2, label='Bimodal threshold')
ax2.set_title('K-means Gap Distribution\n(A vs B, no threshold)', fontweight='bold')
ax2.set_xlabel('Gap')
ax2.legend(fontsize=8)

# Panel 3: CV across conditions
ax3 = fig.add_subplot(2, 3, 3)
cvs_A_all = [r['cv_raw'] for r in results_A]
cvs_B_all = [r['cv_raw'] for r in results_B]
cvs_C_all = [r['cv_weights'] for r in results_C]
ax3.boxplot([cvs_A_all, cvs_B_all, cvs_C_all],
            labels=['A\nconsensus\nraw CV', 'B\nBerman\nraw CV', 'C\nBerman\n+threshold CV'])
ax3.set_title('CV Across Conditions', fontweight='bold')
ax3.set_ylabel('Coefficient of Variation')

# Panel 4: Frequency evolution (B, seed 0)
ax4 = fig.add_subplot(2, 3, 4)
hist_B = results_B[0]['raw_freq_history']
if hist_B:
    cases_t = [h[0] for h in hist_B]
    for p in PRINCIPLES_DEFAULT:
        vals_t = [h[1].get(p, 0) for h in hist_B]
        ax4.plot(cases_t, vals_t, lw=1.5, label=p)
    ax4.set_title('Freq Evolution (Berman)\nSeed 0', fontweight='bold')
    ax4.set_xlabel('Cases')
    ax4.legend(fontsize=6, ncol=2)

# Panel 5: Dip test p-values
ax5 = fig.add_subplot(2, 3, 5)
ax5.scatter(range(len(dip_A)), sorted(dip_A), color='#aec7e8', s=50,
            edgecolor='black', label='A: consensus')
ax5.scatter(range(len(dip_B)), sorted(dip_B), color='#1f77b4', s=50,
            edgecolor='black', label='B: Berman')
ax5.axhline(0.05, color='red', ls='--', lw=2, label='p = 0.05')
ax5.set_xlabel('Seed (sorted by p-value)')
ax5.set_ylabel("Hartigan's dip test p-value")
ax5.set_title('Formal Bimodality Test', fontweight='bold')
ax5.legend(fontsize=8)

# Panel 6: Condition C weights (median seed)
ax6 = fig.add_subplot(2, 3, 6)
cvs_C_arr = np.array([r['cv_weights'] for r in results_C])
med_idx = int(np.argsort(np.abs(cvs_C_arr - np.median(cvs_C_arr)))[0])
best_w = results_C[med_idx]['system_weights']
sorted_w = sorted(best_w.items(), key=lambda x: x[1], reverse=True)
colors_w = ['#2ca02c' if v > 0.01 else '#d62728' for _, v in sorted_w]
ax6.barh([x[0] for x in sorted_w], [x[1] for x in sorted_w],
         color=colors_w, alpha=0.8, edgecolor='black')
ax6.set_title(f'Condition C Weights (median seed)\nCV={cvs_C_arr[med_idx]:.3f}',
              fontweight='bold')

plt.tight_layout()
plt.savefig('/content/exp1_emergence.png')
plt.show()

## Experiment 2 — Sensitivity Analyses

Three design choices in the ABM are potentially arbitrary. We test each one:

**(a) Agent count:** 50, 100, 200. Tests whether the result is a small-sample
artifact or scales with bench size.

**(b) Invocation tendency spread:** Wide (0.95/0.04), medium (0.75/0.20),
narrow (0.65/0.35). Tests whether bimodality requires extreme methodology
differences or emerges even under mild specialisation.

**(c) Selection cutoff percentile:** 25th, 33rd, 50th (median), 67th. Tests
whether the principle-death mechanism's specific cutoff drives the result.

Each sub-experiment runs the Berman fitness condition (B) without threshold,
since that's the condition whose gap we want to be robust. 10 seeds per
configuration, 3000 cases.

In [8]:
# EXPERIMENT 2: SENSITIVITY ANALYSES
# ============================================================================

def experiment_2_sensitivity(n_seeds=10, n_cases=3000):
    print("\n" + "=" * 70)
    print("EXPERIMENT 2: Sensitivity Analyses")
    print("=" * 70)

    all_results = {}

    # --- 2a: Agent count ---
    print("\n  2a: Agent count sensitivity...")
    for n_agents in [50, 100, 200]:
        gaps = []
        for seed in range(n_seeds):
            lsys = LegalSystem(n_agents=n_agents, use_median_threshold=False,
                               fitness_criterion='berman', seed=seed)
            r = lsys.simulate(n_cases=n_cases)
            gaps.append(r['gap_raw'])
        m, lo, hi = bootstrap_ci(gaps)
        bimodal_pct = 100 * sum(1 for g in gaps if g > 0.05) / len(gaps)
        all_results[f'agents_{n_agents}'] = {'gaps': gaps, 'mean': m}
        print(f"    n_agents={n_agents:3d}: gap={m:.4f} [{lo:.4f}, {hi:.4f}]  "
              f"bimodal: {bimodal_pct:.0f}%")

    # --- 2b: Invocation tendency spread ---
    print("\n  2b: Invocation tendency spread...")
    for spread in ['narrow', 'medium', 'wide']:
        gaps = []
        tendencies = make_invocation_tendencies(spread)
        for seed in range(n_seeds):
            lsys = LegalSystem(use_median_threshold=False,
                               fitness_criterion='berman', seed=seed,
                               invocation_tendencies=tendencies)
            r = lsys.simulate(n_cases=n_cases)
            gaps.append(r['gap_raw'])
        m, lo, hi = bootstrap_ci(gaps)
        bimodal_pct = 100 * sum(1 for g in gaps if g > 0.05) / len(gaps)
        all_results[f'spread_{spread}'] = {'gaps': gaps, 'mean': m}
        print(f"    spread={spread:7s}: gap={m:.4f} [{lo:.4f}, {hi:.4f}]  "
              f"bimodal: {bimodal_pct:.0f}%")

    # --- 2c: Selection cutoff (with threshold) ---
    print("\n  2c: Selection cutoff percentile (Condition C)...")
    for pctile in [25, 33, 50, 67]:
        cvs = []
        gaps = []
        for seed in range(n_seeds):
            lsys = LegalSystem(use_median_threshold=True,
                               fitness_criterion='berman', seed=seed,
                               selection_percentile=pctile)
            r = lsys.simulate(n_cases=n_cases)
            cvs.append(r['cv_weights'])
            gaps.append(r['gap_raw'])
        m_cv, lo_cv, hi_cv = bootstrap_ci(cvs)
        survivors = [9 - sum(1 for v in r['system_weights'].values() if v < 0.01)
                     for r in [lsys.simulate(n_cases=n_cases)]]  # re-use last
        all_results[f'cutoff_{pctile}'] = {'cvs': cvs, 'mean_cv': m_cv}
        print(f"    cutoff={pctile:2d}th pctile: cv={m_cv:.4f} [{lo_cv:.4f}, {hi_cv:.4f}]")

    return all_results

sensitivity_results = experiment_2_sensitivity()


EXPERIMENT 2: Sensitivity Analyses

  2a: Agent count sensitivity...
    n_agents= 50: gap=0.5459 [0.4890, 0.6048]  bimodal: 100%
    n_agents=100: gap=0.5602 [0.5267, 0.5930]  bimodal: 100%
    n_agents=200: gap=0.4973 [0.4612, 0.5303]  bimodal: 100%

  2b: Invocation tendency spread...
    spread=narrow : gap=0.0216 [0.0194, 0.0236]  bimodal: 0%
    spread=medium : gap=0.1361 [0.1083, 0.1606]  bimodal: 100%
    spread=wide   : gap=0.4157 [0.3679, 0.4666]  bimodal: 100%

  2c: Selection cutoff percentile (Condition C)...
    cutoff=25th pctile: cv=0.7045 [0.6938, 0.7139]
    cutoff=33th pctile: cv=0.8037 [0.7990, 0.8079]
    cutoff=50th pctile: cv=0.8952 [0.8952, 0.8953]
    cutoff=67th pctile: cv=1.4144 [1.4144, 1.4144]


In [9]:
# EXPERIMENT 2 — VISUALISATION
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Experiment 2: Sensitivity Analyses', fontsize=13, fontweight='bold')

# 2a: Agent count
ax = axes[0]
for key, res in sensitivity_results.items():
    if key.startswith('agents_'):
        n = int(key.split('_')[1])
        m, lo, hi = bootstrap_ci(res['gaps'])
        ax.bar(str(n), m, yerr=[[m-lo],[hi-m]], color='steelblue',
               alpha=0.8, edgecolor='black', capsize=5)
ax.axhline(0.05, color='red', ls='--', lw=1.5, label='Bimodal threshold')
ax.set_xlabel('Agent count')
ax.set_ylabel('K-means gap')
ax.set_title('2a: Agent Count', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

# 2b: Spread
ax = axes[1]
spreads = ['narrow', 'medium', 'wide']
colors_sp = ['#d62728', '#f0ad4e', '#2ca02c']
for j, spread in enumerate(spreads):
    key = f'spread_{spread}'
    if key in sensitivity_results:
        m, lo, hi = bootstrap_ci(sensitivity_results[key]['gaps'])
        ax.bar(j, m, yerr=[[m-lo],[hi-m]], color=colors_sp[j],
               alpha=0.8, edgecolor='black', capsize=5)
ax.axhline(0.05, color='red', ls='--', lw=1.5)
ax.set_xticks(range(len(spreads)))
ax.set_xticklabels([s.capitalize() for s in spreads])
ax.set_xlabel('Invocation tendency spread')
ax.set_ylabel('K-means gap')
ax.set_title('2b: Methodology Spread', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# 2c: Selection cutoff
ax = axes[2]
cutoffs = []
cvs_cut = []
for key, res in sensitivity_results.items():
    if key.startswith('cutoff_'):
        pctile = int(key.split('_')[1])
        cutoffs.append(pctile)
        cvs_cut.append(res['mean_cv'])
if cutoffs:
    ax.plot(cutoffs, cvs_cut, 'o-', color='mediumpurple', linewidth=2, markersize=8)
    ax.set_xlabel('Selection cutoff (percentile)')
    ax.set_ylabel('Weight CV (Condition C)')
    ax.set_title('2c: Selection Cutoff', fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/exp2_sensitivity.png')
plt.show()

## Experiment 3 — Emergent Activation Structure

Notebook 0 showed that correlated activations double robustness. The ABM
generates cases with independent activations, but agents create effective
correlation through their invocation patterns — a textualist who invokes
"text" also tends to invoke "original," creating within-case correlation.

This experiment measures the actual correlation structure in the ABM's output
by tracking which principles co-occur in panel decisions. If the ABM naturally
produces category-correlated patterns (textual principles co-occurring,
moral principles co-occurring), the effective robustness of the emergent
weights is higher than the independent-activation baseline from Notebook 0.

In [10]:
# EXPERIMENT 3: EMERGENT ACTIVATION STRUCTURE
# ============================================================================

def experiment_3_correlation(n_seeds=10, n_cases=3000):
    print("\n" + "=" * 70)
    print("EXPERIMENT 3: Emergent Activation Structure")
    print("  Does the ABM produce realistic principle correlation?")
    print("=" * 70)

    all_corr_matrices = []

    for seed in range(n_seeds):
        lsys = LegalSystem(use_median_threshold=True,
                           fitness_criterion='berman', seed=seed)

        # Track co-invocation across all panel decisions
        n_p = len(PRINCIPLES_DEFAULT)
        co_invoke = np.zeros((n_p, n_p))
        invoke_count = np.zeros(n_p)

        np.random.seed(seed)
        for c in range(n_cases):
            activations = {p: np.random.uniform(-1, 1) for p in PRINCIPLES_DEFAULT}
            panel = np.random.choice(lsys.agents, size=9, replace=False)

            for agent in panel:
                invoked, _ = agent.decide_case(activations, lsys.system_weights)
                inv_idx = [PRINCIPLES_DEFAULT.index(p) for p in invoked
                           if p in PRINCIPLES_DEFAULT]
                for idx in inv_idx:
                    invoke_count[idx] += 1
                for ii in range(len(inv_idx)):
                    for jj in range(ii+1, len(inv_idx)):
                        co_invoke[inv_idx[ii], inv_idx[jj]] += 1
                        co_invoke[inv_idx[jj], inv_idx[ii]] += 1

            # Update system (simplified — just run the full simulation)
            if (c + 1) % 100 == 0:
                lsys.system_weights = lsys._compute_weights()
            if (c + 1) % 500 == 0 and c > 0:
                lsys._apply_selection()

        # Normalise to correlation-like matrix
        for i in range(n_p):
            for j in range(n_p):
                if invoke_count[i] > 0 and invoke_count[j] > 0:
                    co_invoke[i, j] /= np.sqrt(invoke_count[i] * invoke_count[j])

        np.fill_diagonal(co_invoke, 1.0)
        all_corr_matrices.append(co_invoke)

    # Average correlation matrix
    mean_corr = np.mean(all_corr_matrices, axis=0)

    # Categorise: textual (0-3) vs moral (4-8)
    textual_idx = [0, 1, 2, 3]  # text, original, structure, precedent
    moral_idx = [4, 5, 6, 7, 8]  # dignity, equality, liberty, federalism, pragmatic

    within_text = np.mean([mean_corr[i, j] for i in textual_idx
                           for j in textual_idx if i != j])
    within_moral = np.mean([mean_corr[i, j] for i in moral_idx
                            for j in moral_idx if i != j])
    across = np.mean([mean_corr[i, j] for i in textual_idx for j in moral_idx])

    print(f"\n  Co-invocation correlation structure:")
    print(f"    Within textual:  {within_text:.3f}")
    print(f"    Within moral:    {within_moral:.3f}")
    print(f"    Across groups:   {across:.3f}")
    print(f"    Ratio (within/across): {(within_text + within_moral) / 2 / across:.2f}x")

    if (within_text + within_moral) / 2 > across * 1.2:
        print(f"\n  → ABM produces CATEGORY-CORRELATED activation patterns.")
        print(f"    Textual principles co-occur; moral principles co-occur.")
        print(f"    This matches Notebook 0 Experiment 8's 'category_correlated' condition,")
        print(f"    which showed ~2x robustness improvement over independence.")
    else:
        print(f"\n  → ABM does NOT produce strong category correlation.")
        print(f"    The independence assumption in Notebook 0 is appropriate.")

    # Visualise
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(mean_corr, cmap='RdBu_r', vmin=-0.5, vmax=1.0)
    ax.set_xticks(range(len(PRINCIPLES_DEFAULT)))
    ax.set_yticks(range(len(PRINCIPLES_DEFAULT)))
    ax.set_xticklabels(PRINCIPLES_DEFAULT, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(PRINCIPLES_DEFAULT, fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title('ABM Co-Invocation Correlation Matrix\n(Average over seeds)',
                 fontweight='bold')

    # Draw boxes around categories
    for idx_group, color in [(textual_idx, 'blue'), (moral_idx, 'green')]:
        lo_i, hi_i = min(idx_group) - 0.5, max(idx_group) + 0.5
        rect = plt.Rectangle((lo_i, lo_i), hi_i - lo_i, hi_i - lo_i,
                              fill=False, edgecolor=color, linewidth=2.5)
        ax.add_patch(rect)

    plt.tight_layout()
    plt.savefig('/content/exp3_correlation.png')
    plt.show()

    return mean_corr

exp3_corr = experiment_3_correlation()


EXPERIMENT 3: Emergent Activation Structure
  Does the ABM produce realistic principle correlation?

  Co-invocation correlation structure:
    Within textual:  0.601
    Within moral:    0.460
    Across groups:   0.429
    Ratio (within/across): 1.23x

  → ABM produces CATEGORY-CORRELATED activation patterns.
    Textual principles co-occur; moral principles co-occur.
    This matches Notebook 0 Experiment 8's 'category_correlated' condition,
    which showed ~2x robustness improvement over independence.


## Experiment 4 — Cross-Validation Against Notebook 0 Criteria

This is the punchline. We take the converged weights from the ABM's Condition C
(Berman fitness + threshold) and run them through the five testable criteria
from Notebook 0:

1. **Weight uncertainty tolerance ≥ ±0.07** (minimum from Notebook 0 grid)
2. **Berman beats Hart by ≥ +19.1%** (minimum from Notebook 0 Experiment 4)
3. **Emergent threshold in 0.05–0.30** (normatively acceptable range)
4. **Robustness improves under correlation** (Notebook 0 Experiment 8 finding)
5. **Cardinal variation preserved** (weights not binary or uniform)

If the emergent weights pass all five tests, the argument holds end-to-end:
Berman's model works in theory (Notebook 0) *and* its parameters can emerge
from practice (Notebook 1).

In [11]:
# EXPERIMENT 4: CROSS-VALIDATION AGAINST NOTEBOOK 0 CRITERIA
# ============================================================================

def experiment_4_crossvalidation(results_C, n_seeds_to_test=10):
    """
    Take converged weights from ABM Condition C and run Notebook 0's tests.
    """
    print("\n" + "=" * 70)
    print("EXPERIMENT 4: Cross-Validation Against Notebook 0 Criteria")
    print("  Do emergent weights pass the theoretical tests?")
    print("=" * 70)

    criteria_passed = {1: [], 2: [], 3: [], 4: [], 5: []}

    for seed_idx in range(min(n_seeds_to_test, len(results_C))):
        r = results_C[seed_idx]
        emergent_weights = r['system_weights']
        w_vals = np.array(list(emergent_weights.values()))
        p_names = list(emergent_weights.keys())

        print(f"\n  --- Seed {seed_idx} ---")
        print(f"  Emergent weights: {dict(zip(p_names, [f'{v:.3f}' for v in w_vals]))}")

        # CRITERION 1: Weight uncertainty tolerance ≥ 0.07
        # Build a BermanModel with emergent weights
        principles_em = [Principle(p, float(w)) for p, w in emergent_weights.items()
                         if w > 0.001]
        if len(principles_em) < 2:
            print(f"  C1: SKIP — fewer than 2 surviving principles")
            criteria_passed[1].append(False)
        else:
            model_em = BermanModel(principles_em, threshold=0.2)
            rng = np.random.RandomState(seed_idx)
            names_em = [p.name for p in principles_em]
            w_arr = np.array([p.weight for p in principles_em])
            n_test = 300
            n_mc = 40
            sigma_levels = np.linspace(0, 0.50, 16)

            a_mat = rng.uniform(-1, 1, size=(n_test, len(names_em)))
            noise = rng.normal(0, 1, size=(n_test, n_mc, len(names_em)))
            true_forces = a_mat @ w_arr
            true_det = np.abs(true_forces) >= 0.2

            curve = [1.0]
            for sigma in sigma_levels[1:]:
                noisy_w = np.clip(w_arr[None, None, :] + sigma * noise, 0, 1)
                noisy_f = np.sum(noisy_w * a_mat[:, None, :], axis=2)
                noisy_det = np.abs(noisy_f) >= 0.2
                agree = np.mean(noisy_det == true_det[:, None], axis=1)
                curve.append(float(np.mean(agree >= 0.9)))

            crit_sigma = find_crossing(np.array(curve), sigma_levels, 0.80)
            passed = crit_sigma >= 0.07
            criteria_passed[1].append(passed)
            print(f"  C1: σ* = ±{crit_sigma:.3f}  {'PASS ✅' if passed else 'FAIL ❌'} (need ≥0.07)")

        # CRITERION 2: Berman beats Hart by ≥ 19.1%
        if len(principles_em) >= 2:
            rng2 = np.random.RandomState(seed_idx + 1000)
            n_test_hart = 500
            berman_det = 0
            hart_hier_det = 0
            # Use emergent weights for Berman
            # For Hart hierarchical: top 2 weights = tier 1
            sorted_p = sorted(emergent_weights.items(), key=lambda x: x[1], reverse=True)
            tier1 = {p for p, _ in sorted_p[:2]}
            tier2 = {p for p, _ in sorted_p[2:5]}

            for _ in range(n_test_hart):
                acts = {p: rng2.uniform(-1, 1) for p in p_names}
                # Berman
                f_b = sum(emergent_weights[p] * acts[p] for p in p_names)
                if abs(f_b) >= 0.2:
                    berman_det += 1
                # Hart hierarchical
                t1_strong = any(abs(acts[p]) > 0.7 for p in tier1)
                t2_strong = any(abs(acts[p]) > 0.7 for p in tier2)
                t1_mod = any(abs(acts[p]) > 0.4 for p in tier1)
                if t1_strong or (t1_mod and t2_strong):
                    hart_hier_det += 1

            b_rate = berman_det / n_test_hart
            h_rate = hart_hier_det / n_test_hart
            advantage = b_rate - h_rate
            passed = advantage >= 0.191
            criteria_passed[2].append(passed)
            print(f"  C2: Berman={b_rate:.1%} Hart={h_rate:.1%} "
                  f"adv={advantage:+.1%}  {'PASS ✅' if passed else 'FAIL ❌'} (need ≥+19.1%)")

        # CRITERION 3: Threshold in 0.05–0.30
        # Estimate threshold from the force-outcome log
        det_forces = [abs(f) for f, d in r['force_outcome_log'] if d]
        indet_forces = [abs(f) for f, d in r['force_outcome_log'] if not d]
        if det_forces and indet_forces:
            threshold_est = (np.percentile(det_forces, 5) +
                             np.percentile(indet_forces, 95)) / 2
        else:
            threshold_est = np.nan
        passed = 0.05 <= threshold_est <= 0.30 if not np.isnan(threshold_est) else False
        criteria_passed[3].append(passed)
        print(f"  C3: Est. threshold = {threshold_est:.3f}  "
              f"{'PASS ✅' if passed else 'FAIL ❌'} (need 0.05–0.30)")

        # CRITERION 4: Robustness improves under correlation
        # Quick test: independent vs correlated activations with emergent weights
        if len(principles_em) >= 2:
            n_corr_test = 200
            n_mc_c = 30
            sigma_test = 0.10

            # Independent
            a_ind = rng.uniform(-1, 1, size=(n_corr_test, len(names_em)))
            noise_c = rng.normal(0, sigma_test, size=(n_corr_test, n_mc_c, len(names_em)))
            true_f_ind = a_ind @ w_arr
            true_det_ind = np.abs(true_f_ind) >= 0.2
            noisy_w_c = np.clip(w_arr[None, None, :] + noise_c, 0, 1)
            noisy_f_ind = np.sum(noisy_w_c * a_ind[:, None, :], axis=2)
            agree_ind = float(np.mean(
                np.mean((np.abs(noisy_f_ind) >= 0.2) == true_det_ind[:, None], axis=1) >= 0.9))

            # Category-correlated
            n_em = len(names_em)
            half = n_em // 2
            factor1 = rng.uniform(-1, 1, size=(n_corr_test, 1))
            factor2 = rng.uniform(-1, 1, size=(n_corr_test, 1))
            a_corr = np.zeros((n_corr_test, n_em))
            a_corr[:, :half] = np.clip(0.5 * factor1 + 0.5 * rng.uniform(-1, 1, size=(n_corr_test, half)), -1, 1)
            a_corr[:, half:] = np.clip(0.5 * factor2 + 0.5 * rng.uniform(-1, 1, size=(n_corr_test, n_em - half)), -1, 1)
            true_f_corr = a_corr @ w_arr
            true_det_corr = np.abs(true_f_corr) >= 0.2
            noisy_f_corr = np.sum(noisy_w_c * a_corr[:, None, :], axis=2)
            agree_corr = float(np.mean(
                np.mean((np.abs(noisy_f_corr) >= 0.2) == true_det_corr[:, None], axis=1) >= 0.9))

            passed = agree_corr >= agree_ind
            criteria_passed[4].append(passed)
            print(f"  C4: Independent={agree_ind:.1%} Correlated={agree_corr:.1%}  "
                  f"{'PASS ✅' if passed else 'FAIL ❌'} (corr should help)")

        # CRITERION 5: Cardinal variation (not binary, not uniform)
        live_w = w_vals[w_vals > 0.01]
        if len(live_w) >= 2:
            cv_live = float(np.std(live_w) / np.mean(live_w))
            is_binary = np.all((live_w > 0.9) | (live_w < 0.1))
            is_uniform = cv_live < 0.05
            passed = not is_binary and not is_uniform and cv_live > 0.05
            criteria_passed[5].append(passed)
            print(f"  C5: CV of live weights = {cv_live:.3f}  "
                  f"binary={is_binary}  uniform={is_uniform}  "
                  f"{'PASS ✅' if passed else 'FAIL ❌'}")
        else:
            criteria_passed[5].append(False)
            print(f"  C5: FAIL ❌ — fewer than 2 surviving principles")

    # Summary
    print(f"\n  {'='*50}")
    print(f"  CROSS-VALIDATION SUMMARY ({n_seeds_to_test} seeds)")
    print(f"  {'='*50}")
    criterion_names = {
        1: "Weight uncertainty ≥ ±0.07",
        2: "Berman beats Hart ≥ +19.1%",
        3: "Threshold in 0.05–0.30",
        4: "Correlation helps robustness",
        5: "Cardinal weight variation",
    }
    all_pass = True
    for c_num, results in criteria_passed.items():
        pass_pct = 100 * sum(results) / len(results) if results else 0
        status = "✅" if pass_pct >= 80 else "⚠️ " if pass_pct >= 50 else "❌"
        print(f"  {status} C{c_num}: {criterion_names[c_num]:35s} "
              f"{pass_pct:.0f}% pass ({sum(results)}/{len(results)})")
        if pass_pct < 80:
            all_pass = False

    if all_pass:
        print(f"\n  → ALL CRITERIA PASS. Emergent weights satisfy Notebook 0's requirements.")
        print(f"    Berman's model works in theory AND its parameters can emerge from practice.")
    else:
        print(f"\n  → SOME CRITERIA FAIL. See details above.")
        print(f"    The emergent weights do not fully satisfy Notebook 0's requirements.")

    return criteria_passed

criteria = experiment_4_crossvalidation(results_C, n_seeds_to_test=10)


EXPERIMENT 4: Cross-Validation Against Notebook 0 Criteria
  Do emergent weights pass the theoretical tests?

  --- Seed 0 ---
  Emergent weights: {'text': '0.982', 'original': '0.951', 'structure': '0.967', 'precedent': '1.000', 'dignity': '0.000', 'equality': '0.000', 'liberty': '0.000', 'federalism': '0.968', 'pragmatic': '0.000'}
  C1: σ* = ±0.192  PASS ✅ (need ≥0.07)
  C2: Berman=87.4% Hart=72.6% adv=+14.8%  FAIL ❌ (need ≥+19.1%)
  C3: Est. threshold = 0.087  PASS ✅ (need 0.05–0.30)
  C4: Independent=88.0% Correlated=91.5%  PASS ✅ (corr should help)
  C5: CV of live weights = 0.017  binary=True  uniform=True  FAIL ❌

  --- Seed 1 ---
  Emergent weights: {'text': '0.989', 'original': '0.954', 'structure': '0.978', 'precedent': '1.000', 'dignity': '0.000', 'equality': '0.000', 'liberty': '0.000', 'federalism': '0.977', 'pragmatic': '0.000'}
  C1: σ* = ±0.183  PASS ✅ (need ≥0.07)
  C2: Berman=86.2% Hart=73.6% adv=+12.6%  FAIL ❌ (need ≥+19.1%)
  C3: Est. threshold = 0.081  PASS ✅ (ne

In [12]:
# EXPERIMENT 4 — VISUALISATION
# ============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Experiment 4: Cross-Validation Against Notebook 0 Criteria',
             fontsize=13, fontweight='bold')

# Left: Pass rates per criterion
criterion_names = ['C1: Weight\ntolerance', 'C2: Beats\nHart',
                   'C3: Threshold\nrange', 'C4: Corr\nhelps',
                   'C5: Cardinal\nvariation']
pass_rates = []
for c_num in range(1, 6):
    vals = criteria[c_num]
    pass_rates.append(sum(vals) / len(vals) if vals else 0)

colors_c = ['#2ca02c' if r >= 0.8 else '#f0ad4e' if r >= 0.5 else '#d62728'
            for r in pass_rates]
ax1.barh(range(5), pass_rates, color=colors_c, alpha=0.8, edgecolor='black')
ax1.set_yticks(range(5))
ax1.set_yticklabels(criterion_names, fontsize=9)
ax1.set_xlabel('Pass rate')
ax1.set_title('ABM Emergent Weights\nvs Notebook 0 Criteria', fontweight='bold')
ax1.set_xlim(0, 1.1)
ax1.axvline(0.8, color='green', ls='--', lw=1.5, label='80% threshold')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3, axis='x')
for j, rate in enumerate(pass_rates):
    ax1.text(rate + 0.02, j, f'{rate:.0%}', va='center', fontweight='bold')

# Right: Representative emergent weights vs Notebook 0 stipulated
ax = ax2
# Emergent (median seed)
cvs_arr = np.array([r['cv_weights'] for r in results_C])
med_idx = int(np.argsort(np.abs(cvs_arr - np.median(cvs_arr)))[0])
em_w = results_C[med_idx]['system_weights']
# Stipulated (moderate, from Notebook 0)
stip_w = {p: w for p, w in zip(PRINCIPLES_DEFAULT,
          np.linspace(0.50, 0.80, len(PRINCIPLES_DEFAULT))[::-1])}

ps = sorted(em_w.keys(), key=lambda p: stip_w.get(p, 0), reverse=True)
x = np.arange(len(ps))
w = 0.35
ax.barh(x + w/2, [stip_w[p] for p in ps], w,
        label='Stipulated (Nb0)', color='#aec7e8', edgecolor='black')
ax.barh(x - w/2, [em_w[p] for p in ps], w,
        label='Emergent (ABM)', color='#1f77b4', edgecolor='black')
ax.set_yticks(x)
ax.set_yticklabels(ps, fontsize=9)
ax.set_title('Stipulated vs Emergent Weights', fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, 1.1)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('/content/exp4_crossvalidation.png')
plt.show()

## Experiment 5 — SAW Fidelity

Does the ABM's emergent behavior actually match the SAW formula? This is
the test the existing notebook's Shapley experiment should have been.

**Method:** After the ABM converges, generate fresh test cases. For each case,
have the panel of agents decide it (producing a panel-average net force through
their invocation decisions). Separately, compute the SAW prediction using the
emergent weights: F_SAW = Σ(w_i × a_i). Compare the two.

If the correlation is high, the ABM's behavior is well-described by SAW.
If it's low, the ABM has introduced non-linearities (through social learning,
selection, etc.) that SAW can't capture.

In [13]:
# EXPERIMENT 5: SAW FIDELITY
# ============================================================================

def experiment_5_saw_fidelity(n_seeds=10, n_test_cases=500):
    print("\n" + "=" * 70)
    print("EXPERIMENT 5: SAW Fidelity")
    print("  Does the ABM's actual behavior match Σ(w × a)?")
    print("=" * 70)

    all_corrs = []
    all_rmse = []

    for seed in range(n_seeds):
        # Run ABM to convergence
        lsys = LegalSystem(use_median_threshold=True,
                           fitness_criterion='berman', seed=seed)
        r = lsys.simulate(n_cases=3000)
        weights = r['system_weights']

        # Generate fresh test cases
        np.random.seed(seed + 5000)
        abm_forces = []
        saw_forces = []

        for _ in range(n_test_cases):
            activations = {p: np.random.uniform(-1, 1) for p in PRINCIPLES_DEFAULT}
            panel = np.random.choice(lsys.agents, size=9, replace=False)

            # ABM: panel average force
            panel_forces = []
            for agent in panel:
                _, force = agent.decide_case(activations, weights)
                panel_forces.append(force)
            abm_force = np.mean(panel_forces)

            # SAW prediction
            saw_force = sum(weights[p] * activations[p] for p in PRINCIPLES_DEFAULT)

            abm_forces.append(abm_force)
            saw_forces.append(saw_force)

        corr = float(np.corrcoef(abm_forces, saw_forces)[0, 1])
        rmse = float(np.sqrt(np.mean((np.array(abm_forces) - np.array(saw_forces))**2)))
        all_corrs.append(corr)
        all_rmse.append(rmse)

        print(f"  Seed {seed:2d}: corr(ABM, SAW) = {corr:.4f}  RMSE = {rmse:.4f}")

    m_corr, lo_corr, hi_corr = bootstrap_ci(all_corrs)
    m_rmse, lo_rmse, hi_rmse = bootstrap_ci(all_rmse)

    print(f"\n  Summary ({n_seeds} seeds):")
    print(f"    Correlation: {m_corr:.4f} [{lo_corr:.4f}, {hi_corr:.4f}]")
    print(f"    RMSE:        {m_rmse:.4f} [{lo_rmse:.4f}, {hi_rmse:.4f}]")

    if m_corr > 0.95:
        print(f"\n  → Excellent SAW fidelity. The ABM's emergent behavior")
        print(f"    is well-described by the SAW formula with emergent weights.")
    elif m_corr > 0.80:
        print(f"\n  → Good SAW fidelity with some deviation.")
        print(f"    Social learning and selection introduce mild non-linearities.")
    else:
        print(f"\n  → Weak SAW fidelity. The ABM's behavior diverges from SAW.")
        print(f"    This suggests the emergent dynamics are not purely additive.")

    return all_corrs, all_rmse

saw_corrs, saw_rmse = experiment_5_saw_fidelity()


EXPERIMENT 5: SAW Fidelity
  Does the ABM's actual behavior match Σ(w × a)?
  Seed  0: corr(ABM, SAW) = 0.9994  RMSE = 0.0463
  Seed  1: corr(ABM, SAW) = 0.9994  RMSE = 0.0466
  Seed  2: corr(ABM, SAW) = 0.9992  RMSE = 0.0506
  Seed  3: corr(ABM, SAW) = 0.9994  RMSE = 0.0449
  Seed  4: corr(ABM, SAW) = 0.9994  RMSE = 0.0436
  Seed  5: corr(ABM, SAW) = 0.9994  RMSE = 0.0441
  Seed  6: corr(ABM, SAW) = 0.9994  RMSE = 0.0452
  Seed  7: corr(ABM, SAW) = 0.9994  RMSE = 0.0429
  Seed  8: corr(ABM, SAW) = 0.9994  RMSE = 0.0466
  Seed  9: corr(ABM, SAW) = 0.9993  RMSE = 0.0475

  Summary (10 seeds):
    Correlation: 0.9994 [0.9993, 0.9994]
    RMSE:        0.0458 [0.0446, 0.0472]

  → Excellent SAW fidelity. The ABM's emergent behavior
    is well-described by the SAW formula with emergent weights.


In [14]:
# EXPERIMENT 5 — VISUALISATION
# ============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Experiment 5: SAW Fidelity — Does the ABM Match the Formula?',
             fontsize=13, fontweight='bold')

# Left: Correlation distribution
ax1.hist(saw_corrs, bins=max(3, len(saw_corrs)//2),
         color='gold', edgecolor='black', alpha=0.9)
ax1.axvline(np.mean(saw_corrs), color='green', ls='--', lw=2,
            label=f'Mean = {np.mean(saw_corrs):.4f}')
ax1.set_xlabel('Correlation (ABM vs SAW)')
ax1.set_ylabel('Count')
ax1.set_title('SAW Fidelity: corr(ABM force, SAW force)', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Right: RMSE distribution
ax2.hist(saw_rmse, bins=max(3, len(saw_rmse)//2),
         color='lightcoral', edgecolor='black', alpha=0.9)
ax2.axvline(np.mean(saw_rmse), color='red', ls='--', lw=2,
            label=f'Mean = {np.mean(saw_rmse):.4f}')
ax2.set_xlabel('RMSE')
ax2.set_ylabel('Count')
ax2.set_title('Prediction Error', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/exp5_saw_fidelity.png')
plt.show()

## Summary

This notebook tested whether Berman's model parameters can emerge from
simulated legal practice. The key findings:

1. **Emergence under what fitness criterion?** The three-condition comparison
   shows whether bimodal differentiation depends on the selection mechanism
   (consensus-tracking vs. force-magnitude).

2. **Sensitivity.** The results' dependence on agent count, invocation spread,
   and selection cutoff tells us how fragile or robust the emergence finding is.

3. **Realistic correlation.** The ABM's co-invocation patterns show whether
   agents naturally produce the category-correlated activation structures that
   Notebook 0 showed improve robustness.

4. **Cross-validation.** The emergent weights are tested against Notebook 0's
   five criteria. This is the test that connects theoretical model properties
   (Notebook 0) to practice-grounded parameters (Notebook 1).

5. **SAW fidelity.** The ABM's actual behavior is compared to the SAW formula
   to ensure that the emergent dynamics are consistent with Berman's theory.

**What this does not establish:** Whether the specific mechanism (evolutionary
ABM) is the right model of legal practice. Notebook 2 tests alternative
mechanisms (MARL, Hybrid) and shows that the parameters depend on the mechanism —
which is the paper's main theoretical contribution.

In [16]:
# COMPREHENSIVE EXPORT — ALL RESULTS
# ============================================================================

import json, os, shutil
out = "/content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Berman Modeling/Experiment/Computational /Outputs/1"
os.makedirs(out, exist_ok=True)
print(f"Exporting all Notebook 1 results to:\n  {out}\n")

# =====================================================================
# EXPERIMENT 1: Three-Condition Emergence
# =====================================================================
exp1_rows = []
for cond_label, results_list, dip_pvals in [
    ('A_consensus', results_A, dip_A),
    ('B_berman', results_B, dip_B),
]:
    for seed, (r, dp) in enumerate(zip(results_list, dip_pvals)):
        exp1_rows.append({
            'condition': cond_label, 'seed': seed,
            'gap_raw': r['gap_raw'], 'cv_raw': r['cv_raw'],
            'dip_pval': dp,
        })
for seed, r in enumerate(results_C):
    exp1_rows.append({
        'condition': 'C_berman_threshold', 'seed': seed,
        'gap_raw': r.get('gap_raw', float('nan')),
        'cv_raw': r.get('cv_raw', float('nan')),
        'cv_weights': r['cv_weights'],
        'n_survivors': r['n_survivors'],
    })
pd.DataFrame(exp1_rows).to_csv(f'{out}/exp1_emergence.csv', index=False)
print("  exp1_emergence.csv")

# Condition C weights per seed (9 principles × 20 seeds)
weight_rows = []
for seed, r in enumerate(results_C):
    row = {'seed': seed}
    row.update(r['system_weights'])
    weight_rows.append(row)
pd.DataFrame(weight_rows).to_csv(f'{out}/exp1_condition_c_weights.csv', index=False)
print("  exp1_condition_c_weights.csv")

# =====================================================================
# EXPERIMENT 2: Sensitivity Analyses
# =====================================================================
exp2_rows = []
for key, res in sensitivity_results.items():
    if 'gaps' in res:
        m, lo, hi = bootstrap_ci(res['gaps'])
        exp2_rows.append({'config': key, 'metric': 'gap',
                          'mean': m, 'lo': lo, 'hi': hi,
                          'bimodal_pct': 100*sum(1 for g in res['gaps'] if g > 0.05)/len(res['gaps'])})
    if 'cvs' in res:
        m, lo, hi = bootstrap_ci(res['cvs'])
        exp2_rows.append({'config': key, 'metric': 'cv',
                          'mean': m, 'lo': lo, 'hi': hi})
pd.DataFrame(exp2_rows).to_csv(f'{out}/exp2_sensitivity.csv', index=False)
print("  exp2_sensitivity.csv")

# =====================================================================
# EXPERIMENT 3: Correlation Matrix
# =====================================================================
corr_df = pd.DataFrame(exp3_corr, columns=PRINCIPLES_DEFAULT, index=PRINCIPLES_DEFAULT)
corr_df.to_csv(f'{out}/exp3_correlation_matrix.csv')
print("  exp3_correlation_matrix.csv")

# =====================================================================
# EXPERIMENT 4: Cross-Validation
# =====================================================================
exp4_rows = []
for c_num, vals in criteria.items():
    for seed, passed in enumerate(vals):
        exp4_rows.append({'criterion': f'C{c_num}', 'seed': seed, 'passed': int(passed)})
pd.DataFrame(exp4_rows).to_csv(f'{out}/exp4_crossvalidation.csv', index=False)
print("  exp4_crossvalidation.csv")

# Pass rate summary
exp4_summary = []
for c_num, vals in criteria.items():
    rate = sum(vals)/len(vals) if vals else 0
    exp4_summary.append({'criterion': f'C{c_num}', 'pass_rate': rate,
                          'n_pass': sum(vals), 'n_total': len(vals)})
pd.DataFrame(exp4_summary).to_csv(f'{out}/exp4_pass_rates.csv', index=False)
print("  exp4_pass_rates.csv")

# =====================================================================
# EXPERIMENT 5: SAW Fidelity
# =====================================================================
exp5_df = pd.DataFrame({
    'seed': range(len(saw_corrs)),
    'correlation': saw_corrs,
    'rmse': saw_rmse,
})
exp5_df.to_csv(f'{out}/exp5_saw_fidelity.csv', index=False)
print("  exp5_saw_fidelity.csv")

# =====================================================================
# ALL PLOTS
# =====================================================================
plot_dir = f'{out}/plots'
os.makedirs(plot_dir, exist_ok=True)
for png in [f for f in os.listdir('/content') if f.endswith('.png')]:
    shutil.copy2(f'/content/{png}', f'{plot_dir}/{png}')
    print(f"  plots/{png}")

# =====================================================================
# MASTER SUMMARY JSON
# =====================================================================
summary = {
    'experiment_1': {
        'A_mean_gap': float(np.mean([r['gap_raw'] for r in results_A])),
        'A_gap_ci': list(bootstrap_ci([r['gap_raw'] for r in results_A])),
        'B_mean_gap': float(np.mean([r['gap_raw'] for r in results_B])),
        'B_gap_ci': list(bootstrap_ci([r['gap_raw'] for r in results_B])),
        'C_mean_cv_weights': float(np.mean([r['cv_weights'] for r in results_C])),
        'dip_A_reject_count': int(sum(1 for p in dip_A if p < 0.05)),
        'dip_B_reject_count': int(sum(1 for p in dip_B if p < 0.05)),
        'dip_total_seeds': len(dip_A),
        'AB_wilcoxon_p': float(stats.wilcoxon(
            [r['gap_raw'] for r in results_B],
            [r['gap_raw'] for r in results_A], alternative='greater')[1]),
    },
    'experiment_4_crossvalidation': {
        f'C{c}_pass_rate': float(sum(v)/len(v)) if v else 0.0
        for c, v in criteria.items()
    },
    'experiment_5_saw_fidelity': {
        'mean_correlation': float(np.mean(saw_corrs)),
        'correlation_ci': list(bootstrap_ci(saw_corrs)),
        'mean_rmse': float(np.mean(saw_rmse)),
        'rmse_ci': list(bootstrap_ci(saw_rmse)),
    },
}

with open(f'{out}/notebook1_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("  notebook1_summary.json")

all_files = []
for root, dirs, files in os.walk(out):
    for fname in files:
        all_files.append(os.path.join(root, fname))
print(f"\n{'='*50}")
print(f"TOTAL: {len(all_files)} files exported to {out}")
print(f"{'='*50}")

Exporting all Notebook 1 results to:
  /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Berman Modeling/Experiment/Computational /Outputs/1

  exp1_emergence.csv
  exp1_condition_c_weights.csv
  exp2_sensitivity.csv
  exp3_correlation_matrix.csv
  exp4_crossvalidation.csv
  exp4_pass_rates.csv
  exp5_saw_fidelity.csv
  plots/exp4_crossvalidation.png
  plots/exp2_sensitivity.png
  plots/exp1_emergence.png
  plots/exp3_correlation.png
  plots/exp5_saw_fidelity.png
  notebook1_summary.json

TOTAL: 13 files exported to /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Berman Modeling/Experiment/Computational /Outputs/1
